In [1]:
import hashlib
import json
import csv
import os
from rdflib import Graph, URIRef, Literal, RDF, RDFS
from rdflib.namespace import XSD, OWL

In [2]:
URI_PREFIX = "http://w3id.org/COMON/"

EXPERIMENT_FOLDER = "/Users/anakostovska/Dropbox/JSI_active_studies/COMON/data/raw_data/experiment_data"
METADATA_FOLDER   = "/Users/anakostovska/Dropbox/JSI_active_studies/COMON/data/raw_data/metadata/experiment_metadata"
OUTPUT_FOLDER     = "/Users/anakostovska/Dropbox/JSI_active_studies/COMON/data/annotations/RDFannotations/experiment_data"

# Classes
CMOAlgorithmExecution          = URIRef("http://w3id.org/COMON/COMON_000006")
CMOAlgorithmIterationExecution = URIRef("http://w3id.org/COMON/COMON_000011")
CMOSolutionEvaluation          = URIRef("http://w3id.org/COMON/COMON_000012")
CMOPerformanceIndicatorExecution = URIRef("http://w3id.org/COMON/COMON_000007")
CMOPerformanceIndicatorDatum   = URIRef("http://w3id.org/COMON/COMON_000013")

# Properties
realizes                   = URIRef("http://purl.obolibrary.org/obo/OBI_0000308") #object property
has_part                   = URIRef("http://purl.obolibrary.org/obo/BFO_0000051") #object property
has_specified_output       = URIRef("http://purl.obolibrary.org/obo/OBI_0000299") #object property
has_value                  = URIRef("http://w3id.org/ontoopt/has_value")
identifier                 = URIRef("http://purl.org/dc/elements/1.1/identifier")
algorithm_iteration_number = URIRef("http://w3id.org/COMON/COMON_000031")

In [3]:
def string_to_hash(name: str) -> str:
    """MD5 hex digest in uppercase — matches Java DatatypeConverter.printHexBinary."""
    return hashlib.md5(name.encode("utf-8")).hexdigest().upper()


def create_resource(graph: Graph, rdf_type: URIRef, label: str) -> URIRef:
    """Create a resource (or return existing one) with type and label triples."""
    uri = URIRef(URI_PREFIX + string_to_hash(label))
    if (uri, RDF.type, rdf_type) not in graph:
        graph.add((uri, RDF.type, rdf_type))
        graph.add((uri, RDFS.label, Literal(label)))
    return uri

In [4]:
def load_metadata_uris(csv_basename: str):
    """
    Parse the experiment metadata JSON-LD file that corresponds to the CSV.
    Returns (experiment, experiment_impl, experiment_exec, algorithm, algorithm_impl, pi_impls)
    where pi_impls is a dict {"HV": URIRef, "IGDPlus": URIRef, "ICMOP": URIRef, "CV": URIRef}.
    All URIs are reused exactly from the metadata file.
    """
    json_ld_path = os.path.join(METADATA_FOLDER, csv_basename.lower() + "-LD.json")
    if not os.path.exists(json_ld_path):
        raise FileNotFoundError(f"No JSON-LD metadata found for '{csv_basename}'")

    with open(json_ld_path, encoding="utf-8") as f:
        data = json.load(f)

    comon_prefix = data["@context"]["COMON"]  # "http://w3id.org/COMON/"

    def expand(id_str: str) -> str:
        if id_str.startswith("COMON:"):
            return comon_prefix + id_str[6:]
        return id_str

    experiment = experiment_impl = experiment_exec = None
    algorithm = algorithm_impl = None
    pi_impls = {}

    for node in data["@graph"]:
        ntype = node.get("@type")
        node_id = node.get("@id")
        if not node_id:
            continue
        uri = URIRef(expand(node_id))

        if ntype == "CMO_experiment":
            experiment = uri
            inputs = node.get("has-specified-input", [])
            if isinstance(inputs, dict):
                inputs = [inputs]
            for inp in inputs:
                inp_uri = expand(inp["@id"])
                if inp_uri.endswith("_algorithm"):
                    algorithm = URIRef(inp_uri)
                    break

        elif ntype == "CMO_experiment_implementation":
            experiment_impl = uri
            uses = node.get("uses", [])
            if isinstance(uses, dict):
                uses = [uses]
            for use in uses:
                use_uri = expand(use["@id"])
                if use_uri.endswith("_algorithm_implementation"):
                    algorithm_impl = URIRef(use_uri)
                elif "_Hypervolume_implementation" in use_uri:
                    pi_impls["HV"] = URIRef(use_uri)
                elif "_IGDPlus_implementation" in use_uri:
                    pi_impls["IGDPlus"] = URIRef(use_uri)
                elif "_ICMOP_implementation" in use_uri:
                    pi_impls["ICMOP"] = URIRef(use_uri)
                elif "_TotalConstraintViolation_implementation" in use_uri:
                    pi_impls["CV"] = URIRef(use_uri)

        elif ntype == "CMO_experiment_execution":
            experiment_exec = uri

    return experiment, experiment_impl, experiment_exec, algorithm, algorithm_impl, pi_impls

In [5]:
def add_performance_value(
    graph: Graph,
    solution_evaluation: URIRef,
    label: str,
    value: str,
    iteration_key: str,
    eval_: str,
    pi_impl: URIRef = None,
):
    if not value or not value.strip():
        return

    execution = create_resource(
        graph,
        CMOPerformanceIndicatorExecution,
        f"{iteration_key}_eval-{eval_}_{label}-performance-indicator-execution",
    )
    graph.add((solution_evaluation, has_part, execution))
    if pi_impl is not None:
        graph.add((execution, realizes, pi_impl))

    datum = create_resource(
        graph,
        CMOPerformanceIndicatorDatum,
        f"{iteration_key}_eval-{eval_}_{label}-performance-indicator-datum",
    )
    graph.add((execution, has_specified_output, datum))
    graph.add((datum, has_value, Literal(round(float(value), 6), datatype=XSD.double)))

In [6]:
def process_experiment_data_csv(csv_path: str) -> Graph:
    graph = Graph()

    # Declare property types so Protégé shows them in the correct tabs
    for prop in [realizes, has_part, has_specified_output]:
        graph.add((prop, RDF.type, OWL.ObjectProperty))
    graph.add((has_value, RDF.type, OWL.DatatypeProperty))
    graph.add((identifier, RDF.type, OWL.AnnotationProperty))
    graph.add((algorithm_iteration_number, RDF.type, OWL.AnnotationProperty))

    csv_basename = os.path.basename(csv_path).replace(".csv", "")

    # Load shared URIs directly from the corresponding JSON-LD metadata file
    experiment, experiment_impl, experiment_exec, algorithm, algorithm_impl, pi_impls = \
        load_metadata_uris(csv_basename)

    algorithm_execution_cache = {}  # run_key -> URIRef
    iteration_execution_cache = {}  # iteration_key -> URIRef

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header

        for row in reader:
            eval_    = row[0].strip()
            gen      = row[1].strip()
            hv       = row[2].strip()
            igd_plus = row[3].strip()
            icmop    = row[4].strip()
            cv       = row[5].strip()
            run      = row[6].strip()

            run_int  = int(float(run))
            gen_int  = int(float(gen))
            eval_int = int(float(eval_))

            run_key       = f"{csv_basename}_run-{run}"
            iteration_key = f"{run_key}_iteration-{gen}"

            # Algorithm execution (cached per run)
            if run_key not in algorithm_execution_cache:
                alg_exec = create_resource(graph, CMOAlgorithmExecution, f"{run_key}_algorithm-execution")
                graph.add((alg_exec, identifier, Literal(run_int)))
                graph.add((alg_exec, realizes,   algorithm_impl))
                graph.add((experiment_exec, has_part, alg_exec))
                algorithm_execution_cache[run_key] = alg_exec
            else:
                alg_exec = algorithm_execution_cache[run_key]

            # Iteration execution (cached per run + generation)
            if iteration_key not in iteration_execution_cache:
                iter_exec = create_resource(graph, CMOAlgorithmIterationExecution,
                                            f"{iteration_key}_algorithm-iteration-execution")
                graph.add((iter_exec, algorithm_iteration_number, Literal(gen_int)))
                graph.add((alg_exec, has_part, iter_exec))
                iteration_execution_cache[iteration_key] = iter_exec
            else:
                iter_exec = iteration_execution_cache[iteration_key]

            # Solution evaluation
            sol_eval = create_resource(graph, CMOSolutionEvaluation,
                                       f"{iteration_key}_eval-{eval_}_solution_evaluation")
            graph.add((sol_eval,  identifier, Literal(eval_int)))
            graph.add((iter_exec, has_part,   sol_eval))

            # Performance values — PI execution realizes the corresponding PI implementation
            add_performance_value(graph, sol_eval, "HV",      hv,       iteration_key, eval_, pi_impls.get("HV"))
            add_performance_value(graph, sol_eval, "IGDPlus", igd_plus, iteration_key, eval_, pi_impls.get("IGDPlus"))
            add_performance_value(graph, sol_eval, "ICMOP",   icmop,    iteration_key, eval_, pi_impls.get("ICMOP"))
            add_performance_value(graph, sol_eval, "CV",      cv,       iteration_key, eval_, pi_impls.get("CV"))

    return graph

In [7]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for csv_filename in sorted(os.listdir(EXPERIMENT_FOLDER)):
    if not csv_filename.lower().endswith(".csv"):
        continue
    # if "gde3_mw1_eval_pymoode" in csv_filename.lower():
    #     continue

    csv_path = os.path.join(EXPERIMENT_FOLDER, csv_filename)
    print(f"Reading: {csv_filename}")

    g = process_experiment_data_csv(csv_path)
    print(f"Triples: {len(g)}")


    output_name = csv_filename.replace(".csv", "").lower()
    output_path = os.path.join(OUTPUT_FOLDER, f"experiment_data_{output_name}.rdf")
    g.serialize(destination=output_path, format="xml")
    print(f"Saved:   {output_path}\n")

Reading: bico_cobi1_final_platemo.csv
Triples: 681
Saved:   /Users/anakostovska/Dropbox/JSI_active_studies/COMON/data/annotations/RDFannotations/experiment_data/experiment_data_bico_cobi1_final_platemo.rdf

Reading: bico_cre31_final_platemo.csv
Triples: 681
Saved:   /Users/anakostovska/Dropbox/JSI_active_studies/COMON/data/annotations/RDFannotations/experiment_data/experiment_data_bico_cre31_final_platemo.rdf

Reading: bico_mw1_final_platemo.csv
Triples: 649
Saved:   /Users/anakostovska/Dropbox/JSI_active_studies/COMON/data/annotations/RDFannotations/experiment_data/experiment_data_bico_mw1_final_platemo.rdf

Reading: ccmo_cobi1_gen_platemo.csv
Triples: 69361
Saved:   /Users/anakostovska/Dropbox/JSI_active_studies/COMON/data/annotations/RDFannotations/experiment_data/experiment_data_ccmo_cobi1_gen_platemo.rdf

Reading: ccmo_cre31_gen_platemo.csv


KeyboardInterrupt: 